In [1]:
import os
import numpy as np
import pandas as pd

OUTPUT_DIR = "star_schema_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def main():
    # ==========================================================
    # Load Source Dataset
    # ==========================================================
    print("Loading source dataset...")

    df = pd.read_csv("Skyscope_Analytics_Cleaned_Full.csv")

    print(f"Loaded {len(df):,} records.")

    # Required columns validation
    required_cols = [
        "Airline",
        "Operating_Airline",
        "IATA_Code_Marketing_Airline",
        "IATA_Code_Operating_Airline",
        "Origin",
        "Dest",
        "FlightDate",
        "DepDelayMinutes",
        "ArrDelayMinutes",
        "AirTime",
        "Distance",
        "IsDelayed",
        "FlightStatus",
    ]

    missing = [col for col in required_cols if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # ==========================================================
    # Convert Flight Date
    # ==========================================================
    print("Converting flight dates...")

    df["FlightDate_Parsed"] = pd.to_datetime(
        df["FlightDate"],
        errors="coerce"
    )

    # ==========================================================
    # Build Airline Dimension
    # ==========================================================
    print("Building Dim_Airline...")

    dim_airline = (
        df[
            [
                "Airline",
                "Operating_Airline",
                "IATA_Code_Marketing_Airline",
                "IATA_Code_Operating_Airline",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    dim_airline.insert(0, "Airline_ID", dim_airline.index + 1)

    dim_airline = dim_airline.rename(
        columns={
            "Airline": "Airline_Name",
            "IATA_Code_Marketing_Airline": "Marketing_Code",
            "IATA_Code_Operating_Airline": "Operating_Code",
        }
    )

    print(f"Dim_Airline created with {len(dim_airline):,} records.")

    # ==========================================================
    # Build Airport Dimension
    # ==========================================================
    print("Building Dim_Airport...")

    airport_codes = pd.unique(
        pd.concat([df["Origin"], df["Dest"]]).dropna()
    )

    dim_airport = pd.DataFrame(
        {
            "Airport_ID": range(1, len(airport_codes) + 1),
            "Airport_Code": airport_codes,
        }
    )

    print(f"Dim_Airport created with {len(dim_airport):,} records.")

    # ==========================================================
    # Build Date Dimension
    # ==========================================================
    print("Building Dim_Date...")

    unique_dates = (
        df["FlightDate_Parsed"]
        .dropna()
        .drop_duplicates()
        .sort_values()
    )

    dim_date = pd.DataFrame({"Full_Date": unique_dates})

    dim_date["Date_Key"] = (
        dim_date["Full_Date"]
        .dt.strftime("%Y%m%d")
        .astype(int)
    )

    dim_date["Flight_Year"] = dim_date["Full_Date"].dt.year
    dim_date["Flight_Month"] = dim_date["Full_Date"].dt.month
    dim_date["Flight_Month_Name"] = dim_date["Full_Date"].dt.month_name()
    dim_date["Flight_Day"] = dim_date["Full_Date"].dt.day
    dim_date["Day_Of_Week_Name"] = dim_date["Full_Date"].dt.day_name()

    dim_date = dim_date[
        [
            "Date_Key",
            "Full_Date",
            "Flight_Year",
            "Flight_Month",
            "Flight_Month_Name",
            "Flight_Day",
            "Day_Of_Week_Name",
        ]
    ].reset_index(drop=True)

    print(f"Dim_Date created with {len(dim_date):,} records.")

    # ==========================================================
    # Build Fact Table
    # ==========================================================
    print("Building Fact_Flights...")

    fact = df.copy()

    # Map Airline_ID
    fact = fact.merge(
        dim_airline,
        left_on=[
            "Airline",
            "Operating_Airline",
            "IATA_Code_Marketing_Airline",
            "IATA_Code_Operating_Airline",
        ],
        right_on=[
            "Airline_Name",
            "Operating_Airline",
            "Marketing_Code",
            "Operating_Code",
        ],
        how="inner",
    )

    # Map Origin Airport
    fact = fact.merge(
        dim_airport.rename(
            columns={
                "Airport_ID": "Origin_Airport_ID",
                "Airport_Code": "Origin",
            }
        ),
        on="Origin",
        how="inner",
    )

    # Map Destination Airport
    fact = fact.merge(
        dim_airport.rename(
            columns={
                "Airport_ID": "Dest_Airport_ID",
                "Airport_Code": "Dest",
            }
        ),
        on="Dest",
        how="inner",
    )

    # Map Date Key
    fact = fact.merge(
        dim_date[["Date_Key", "Full_Date"]],
        left_on="FlightDate_Parsed",
        right_on="Full_Date",
        how="inner",
    )

    # Preserve delay flag only for completed flights
    fact["IsDelayed_Final"] = np.where(
        fact["FlightStatus"] == "Completed",
        fact["IsDelayed"],
        np.nan,
    )

    fact_flights = pd.DataFrame(
        {
            "Date_Key": fact["Date_Key"],
            "Airline_ID": fact["Airline_ID"],
            "Origin_Airport_ID": fact["Origin_Airport_ID"],
            "Dest_Airport_ID": fact["Dest_Airport_ID"],
            "DepDelayMinutes": pd.to_numeric(
                fact["DepDelayMinutes"],
                errors="coerce",
            ),
            "ArrDelayMinutes": pd.to_numeric(
                fact["ArrDelayMinutes"],
                errors="coerce",
            ),
            "AirTime": pd.to_numeric(
                fact["AirTime"],
                errors="coerce",
            ),
            "Distance": pd.to_numeric(
                fact["Distance"],
                errors="coerce",
            ),
            "IsDelayed": fact["IsDelayed_Final"],
            "FlightStatus": fact["FlightStatus"],
        }
    )

    fact_flights.insert(
        0,
        "Flight_Fact_ID",
        range(1, len(fact_flights) + 1),
    )

    print(
        f"Fact_Flights created with {len(fact_flights):,} records "
        f"from {len(df):,} source records."
    )

    skipped = len(df) - len(fact_flights)

    if skipped > 0:
        print(f"{skipped:,} records were excluded during dimension mapping.")

    # ==========================================================
    # Export CSV Files
    # ==========================================================
    print("Exporting dimension and fact tables...")

    dim_airline.to_csv(
        f"{OUTPUT_DIR}/Dim_Airline.csv",
        index=False,
    )

    dim_airport.to_csv(
        f"{OUTPUT_DIR}/Dim_Airport.csv",
        index=False,
    )

    dim_date.to_csv(
        f"{OUTPUT_DIR}/Dim_Date.csv",
        index=False,
    )

    fact_flights.to_csv(
        f"{OUTPUT_DIR}/Fact_Flights.csv",
        index=False,
    )

    print("Export completed successfully.")
    print(f"Output directory: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Loading source dataset...
Loaded 4,078,318 records.
Converting flight dates...
Building Dim_Airline...
Dim_Airline created with 27 records.
Building Dim_Airport...
Dim_Airport created with 375 records.
Building Dim_Date...
Dim_Date created with 212 records.
Building Fact_Flights...
Fact_Flights created with 4,078,318 records from 4,078,318 source records.
Exporting dimension and fact tables...
Export completed successfully.
Output directory: star_schema_output
